In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [33]:
# 1. LOAD DATA
df = pd.read_csv("sales_data.csv")

In [34]:
# 2. FEATURE ENGINEERING

# A. Encode Data Kategorikal (Profil)
le = LabelEncoder()
cols_to_encode = ['product_category_name', 'seller_state', 'seller_city']
for col in cols_to_encode:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

# B. Buat Fitur Baru: "Price Segment"
# Mengubah avg_order_value (angka) menjadi Kategori (0=Murah, 1=Sedang, 2=Mahal)
# Kita pakai qcut untuk membagi data menjadi 3 bagian sama besar
df['price_segment'] = pd.qcut(df['avg_order_value'], q=3, labels=[0, 1, 2]).astype(int)

# TARGET: High vs Low Performance
threshold = df['total_sales'].median()
df['performance_tier'] = (df['total_sales'] > threshold).astype(int)

In [35]:
# 3. PILIH FITUR
# Kita gabungin Seberapa Rajin (Orders) + Level Harga Barang (Price Segment) + Profil (City/Cat)
features_final = [
    'total_orders',           # Aktivitas (Volume)
    'price_segment',          # Level Harga (Value)
    'product_category_name',  # Jenis Barang
    'seller_state'            # Lokasi
]

X = df[features_final]
y = df['performance_tier']

In [36]:
# 4. SPLIT DATA
# test_size=0.2 artinya 20% data disisihkan untuk ujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"training data: {X_train.shape[0]} baris")
print(f"test data: {X_test.shape[0]} baris")

training data: 36586 baris
test data: 9147 baris


In [37]:
# 5. TRAINING MODEL
# max_depth=10, biar ckup detail nangkep pola gabungan Orders + Price Segment
model = DecisionTreeClassifier(max_depth=10, random_state=42)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, random_state=42)

In [38]:
# 6. EVALUASI
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Akurasi Model: {accuracy * 100:.2f}%")
print("-" * 40)
print(classification_report(y_test, y_pred))

print("-" * 40)
print("Faktor Penentu (Feature Importance):")
importances = pd.Series(model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

Akurasi Model: 92.65%
----------------------------------------
              precision    recall  f1-score   support

           0       0.95      0.90      0.92      4589
           1       0.91      0.95      0.93      4558

    accuracy                           0.93      9147
   macro avg       0.93      0.93      0.93      9147
weighted avg       0.93      0.93      0.93      9147

----------------------------------------
Faktor Penentu (Feature Importance):
price_segment            0.515720
total_orders             0.462225
product_category_name    0.015367
seller_state             0.006689
dtype: float64
